In [1]:
import sqlite3

In [2]:
connection = sqlite3.connect("coffee_shop.db")

In [3]:
print("Connected:", connection)

Connected: <sqlite3.Connection object at 0x0000024C7ECCFC40>


In [4]:
connection.close()

In [5]:
import sqlite3

with sqlite3.connect("coffee_shop.db") as connection:
    print("Database is open inside this block.")

Database is open inside this block.


In [6]:
with sqlite3.connect("coffee_shop.db") as connection:
    cursor = connection.cursor()



In [8]:
cursor.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name  TEXT NOT NULL,
            last_name   TEXT NOT NULL,
            email       TEXT UNIQUE,
            city        TEXT,
            state       TEXT
        );
    """)

connection.commit

<function Connection.commit()>

In [9]:
with sqlite3.connect("coffee_shop.db") as connection:
    cursor = connection.cursor()

    cursor.execute("""
        INSERT INTO customers (first_name, last_name, email, city, state)
        VALUES (?, ?, ?, ?, ?);
    """, ("Ana", "Lee", "ana.lee@example.com", "Louisville", "KY"))

    connection.commit()

In [10]:
rows = [
    ("Chris", "Nguyen", "chris.nguyen@example.com", "Lexington", "KY"),
    ("Maria", "Soto", "maria.soto@example.com", "Nashville", "TN"),
    ("Derek", "Wells", "derek.wells@example.com", "Cincinnati", "OH"),
]

with sqlite3.connect("coffee_shop.db") as connection:
    cursor = connection.cursor()
    cursor.executemany("""
        INSERT INTO customers (first_name, last_name, email, city, state)
        VALUES (?, ?, ?, ?, ?);
    """, rows)
    connection.commit()

In [11]:
with sqlite3.connect("coffee_shop.db") as connection:
    cursor = connection.cursor()
    cursor.execute("""
        SELECT customer_id, first_name, last_name, city, state
        FROM customers
        ORDER BY last_name, first_name;
    """)

    results = cursor.fetchall()
    for row in results:
        print(row)

(1, 'Ana', 'Lee', 'Louisville', 'KY')
(2, 'Chris', 'Nguyen', 'Lexington', 'KY')
(3, 'Maria', 'Soto', 'Nashville', 'TN')
(4, 'Derek', 'Wells', 'Cincinnati', 'OH')


In [12]:
with sqlite3.connect("coffee_shop.db") as connection:
    cursor = connection.cursor()
    cursor.execute("""
        UPDATE customers
        SET city = ?, state = ?
        WHERE email = ?;
    """, ("Bloomington", "IN", "ana.lee@example.com"))
    connection.commit()
    print("Rows updated:", cursor.rowcount)

Rows updated: 1


In [13]:
with sqlite3.connect("coffee_shop.db") as connection:
    try:
        cursor = connection.cursor()

        cursor.execute("""
            INSERT INTO customers (first_name, last_name, email, city, state)
            VALUES (?, ?, ?, ?, ?);
        """, ("Jin", "Park", "jin.park@example.com", "Indianapolis", "IN"))

        cursor.execute("""
            UPDATE customers
            SET state = ?
            WHERE email = ?;
        """, ("IN", "maria.soto@example.com"))

        connection.commit()
        print("Both actions completed successfully!")

    except Exception as e:
        connection.rollback()
        print("Transaction failed and rolled back:", e)

Both actions completed successfully!


In [14]:
with sqlite3.connect("coffee_shop.db") as connection:
    connection.execute("PRAGMA foreign_keys = ON;")
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS orders (
            order_id    INTEGER PRIMARY KEY AUTOINCREMENT,
            customer_id INTEGER NOT NULL,
            order_date  TEXT NOT NULL,
            total_cents INTEGER NOT NULL,
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        );
    """)
    connection.commit()

    # Insert a sample order for Ana
    cursor.execute("SELECT customer_id FROM customers WHERE email = ?;", ("ana.lee@example.com",))
    ana_id = cursor.fetchone()[0]

    cursor.execute("""
        INSERT INTO orders (customer_id, order_date, total_cents)
        VALUES (?, DATE('now'), ?);
    """, (ana_id, 2595))
    connection.commit()